# 09 — Rank-Based Latest Prediction Snapshot

## Purpose

This notebook creates version 2 of the MarketGuard latest prediction snapshot.

Notebook 07 is preserved as the original implementation, where fixed
downside-probability bands control the combined opportunity-risk class.

Notebook 08 showed that the downside model performs more reliably as a
cross-sectional risk-ranking model. Version 2 therefore uses relative downside
rank for the primary combined classification while retaining the fixed risk
band as a descriptive alert.

## Version Lineage

| Version | Notebook | Primary risk input |
|---|---|---|
| v1 | `07_latest_prediction_snapshot.ipynb` | Fixed downside-probability bands |
| Validation | `08_historical_signal_backtest.ipynb` | Historical fixed-versus-rank comparison |
| v2 | `09_rank_based_prediction_snapshot.ipynb` | Cross-sectional downside-risk rank |

## Risk Views Retained

Version 2 keeps three separate downside-risk views:

| Field | Purpose |
|---|---|
| `downside_risk_band` | Fixed score-based descriptive alert |
| `downside_risk_quintile` | Detailed relative risk from Q1 to Q5 |
| `relative_downside_risk` | Higher/Lower group used for the combined class |

## Main Classification Logic

For the 90 prediction-ready stocks:

### Opportunity

| Outperform rank | Opportunity tier |
|---:|---|
| 1–18 | High Opportunity |
| 19–45 | Moderate Opportunity |
| 46–90 | Low Opportunity |

### Relative Downside Risk

| Downside-risk rank | Detailed risk level | Combined risk group |
|---:|---|---|
| 1–18 | Q1 — Highest Risk | Higher Relative Risk |
| 19–36 | Q2 — High Risk | Higher Relative Risk |
| 37–45 | Upper part of Q3 — Medium Risk | Higher Relative Risk |
| 46–54 | Lower part of Q3 — Medium Risk | Lower Relative Risk |
| 55–72 | Q4 — Lower Risk | Lower Relative Risk |
| 73–90 | Q5 — Lowest Risk | Lower Relative Risk |

The five quintiles provide detailed interpretation. The broader 50/50 split is
used only to create six clear and historically testable combined classes.

## Historical Evidence

In the 2025+ backtest, the rank-based Attractive Risk-Reward group versus the
Unfavourable Risk-Reward group showed:

- No reliable final-return advantage
- 1.84 percentage points shallower worst-path returns
- 13.70 percentage points fewer 5% downside events
- 9.60 percentage points fewer 10% downside events

The system is therefore positioned as a risk-intelligence and screening
framework, not as a proven return-generating strategy.

### Imports, project paths, and v2 output directory.

In [ ]:
Imports, project paths, and v2 output directory

from pathlib import Path
from datetime import datetime, timezone
import json
import math

import joblib
import numpy as np
import pandas as pd
from IPython.display import display


# Resolve the repository root.
# This works whether Jupyter starts from the project root or notebooks folder.
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# Input files — unchanged from Notebook 07

FEATURE_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "targets"
    / "stock_features_with_targets_v1.parquet"
)

OUTPERFORM_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "best_random_forest_outperform_nifty50_20d_v1.joblib"
)

DOWNSIDE_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "random_forest_downside_10pct_20d_v1.joblib"
)


# Separate v2 output location.
# Notebook 07 and all v1 files remain untouched.
OUTPUT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "rank_based_prediction_snapshot"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Notebook/version identifiers used later in metadata and filenames
NOTEBOOK_VERSION = "09"
SNAPSHOT_VERSION = "v2"
CLASSIFICATION_METHOD = "rank_based_relative_downside_risk"


required_paths = {
    "Feature dataset": FEATURE_DATA_PATH,
    "Outperform model": OUTPERFORM_MODEL_PATH,
    "Downside model": DOWNSIDE_MODEL_PATH,
}


print("Project root:")
print(PROJECT_ROOT)

print("\nRequired input files:")
for label, path in required_paths.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{label:<20} [{status}] {path}")

print("\nV2 output directory:")
print(OUTPUT_DIR)


missing_paths = [
    str(path)
    for path in required_paths.values()
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_paths)
    )

print("\nPath validation passed.")

Project root:
E:\Projects\marketguard-india

Required input files:
Feature dataset      [FOUND] E:\Projects\marketguard-india\data\interim\targets\stock_features_with_targets_v1.parquet
Outperform model     [FOUND] E:\Projects\marketguard-india\models\best_random_forest_outperform_nifty50_20d_v1.joblib
Downside model       [FOUND] E:\Projects\marketguard-india\models\random_forest_downside_10pct_20d_v1.joblib

V2 output directory:
E:\Projects\marketguard-india\reports\rank_based_prediction_snapshot

Path validation passed.


### Load dataset and trained models, then validate model features.

In [3]:
#  Load dataset and trained models

feature_data = pd.read_parquet(FEATURE_DATA_PATH)

outperform_model = joblib.load(OUTPERFORM_MODEL_PATH)
downside_model = joblib.load(DOWNSIDE_MODEL_PATH)


def extract_feature_names(fitted_model) -> list[str]:
    """
    Extract the fitted input feature names from a saved sklearn estimator
    or Pipeline.

    The saved feature order must be reused exactly during inference.
    """

    if hasattr(fitted_model, "feature_names_in_"):
        return list(fitted_model.feature_names_in_)

    if hasattr(fitted_model, "named_steps"):
        for step in fitted_model.named_steps.values():
            if hasattr(step, "feature_names_in_"):
                return list(step.feature_names_in_)

    raise AttributeError(
        "Could not find feature_names_in_ in the fitted model or pipeline."
    )


outperform_features = extract_feature_names(outperform_model)
downside_features = extract_feature_names(downside_model)


# Confirm both models use the same ordered feature list.
same_feature_order = outperform_features == downside_features

if not same_feature_order:
    raise ValueError(
        "The outperform and downside models do not use the same ordered "
        "feature list."
    )


MODEL_FEATURES = outperform_features


# Check that every required feature exists in the source dataset.
missing_dataset_features = [
    feature
    for feature in MODEL_FEATURES
    if feature not in feature_data.columns
]

if missing_dataset_features:
    raise KeyError(
        "The feature dataset is missing required model columns:\n"
        + "\n".join(missing_dataset_features)
    )


# Standardize the date column before selecting the latest observations.
feature_data["date"] = pd.to_datetime(feature_data["date"])


print("Dataset loaded:")
print(f"Shape: {feature_data.shape}")
print(
    "Date range:",
    feature_data["date"].min().date(),
    "to",
    feature_data["date"].max().date(),
)

if "yf_ticker" in feature_data.columns:
    print(
        "Unique stocks:",
        feature_data["yf_ticker"].nunique(),
    )

print("\nModels loaded:")
print("Outperform model:", type(outperform_model).__name__)
print("Downside model:", type(downside_model).__name__)

print("\nModel feature validation:")
print("Outperform feature count:", len(outperform_features))
print("Downside feature count:", len(downside_features))
print("Identical feature order:", same_feature_order)
print("Missing required dataset features:", len(missing_dataset_features))

print("\nDataset and model validation passed.")

Dataset loaded:
Shape: (357370, 200)
Date range: 2010-01-04 to 2026-07-14
Unique stocks: 100

Models loaded:
Outperform model: Pipeline
Downside model: Pipeline

Model feature validation:
Outperform feature count: 79
Downside feature count: 79
Identical feature order: True
Missing required dataset features: 0

Dataset and model validation passed.


### Build the latest stock cross-section and assess data readiness.

In [4]:
# Select latest stock rows and assess prediction readiness

required_readiness_columns = [
    "yf_ticker",
    "date",
    "model_ready_v1",
]

missing_readiness_columns = [
    column
    for column in required_readiness_columns
    if column not in feature_data.columns
]

if missing_readiness_columns:
    raise KeyError(
        "Required readiness columns are missing:\n"
        + "\n".join(missing_readiness_columns)
    )


# Sort chronologically and keep the most recent observation for each stock.
latest_snapshot_base = (
    feature_data
    .sort_values(["yf_ticker", "date"])
    .groupby("yf_ticker", group_keys=False)
    .tail(1)
    .copy()
    .reset_index(drop=True)
)


# Count missing values across the exact 79 features expected by the models.
latest_snapshot_base["missing_model_feature_count"] = (
    latest_snapshot_base[MODEL_FEATURES]
    .isna()
    .sum(axis=1)
)

latest_snapshot_base["has_complete_model_features"] = (
    latest_snapshot_base["missing_model_feature_count"].eq(0)
)


# Normal prediction readiness requires:
# 1. Sufficient historical eligibility from the feature pipeline.
# 2. No missing model-input features.
latest_snapshot_base["prediction_ready"] = (
    latest_snapshot_base["model_ready_v1"]
    .fillna(False)
    .astype(bool)
    &
    latest_snapshot_base["has_complete_model_features"]
)


def assign_data_confidence(row: pd.Series) -> str:
    """
    Describe the quality and historical completeness of the model inputs.

    This is data confidence, not the model's statistical confidence.
    """

    historically_ready = bool(row["model_ready_v1"])
    complete_features = bool(row["has_complete_model_features"])

    if historically_ready and complete_features:
        return "Normal Confidence"

    if not historically_ready and complete_features:
        return "Limited History"

    if not historically_ready and not complete_features:
        return "Limited History + Missing Features"

    return "Missing Features"


latest_snapshot_base["data_confidence"] = (
    latest_snapshot_base.apply(
        assign_data_confidence,
        axis=1,
    )
)


# Cross-sectional validation
latest_row_count = len(latest_snapshot_base)
latest_unique_stocks = latest_snapshot_base["yf_ticker"].nunique()
latest_unique_dates = latest_snapshot_base["date"].nunique()
latest_date = latest_snapshot_base["date"].max()

ready_stock_count = int(
    latest_snapshot_base["prediction_ready"].sum()
)

limited_stock_count = int(
    (~latest_snapshot_base["prediction_ready"]).sum()
)


print("Latest cross-section:")
print("Rows:", latest_row_count)
print("Unique stocks:", latest_unique_stocks)
print("Unique latest dates:", latest_unique_dates)
print("Latest date:", latest_date.date())

print("\nPrediction readiness:")
print("Prediction-ready stocks:", ready_stock_count)
print("Limited-confidence stocks:", limited_stock_count)

print("\nData-confidence summary:")
display(
    latest_snapshot_base["data_confidence"]
    .value_counts()
    .rename_axis("data_confidence")
    .reset_index(name="stock_count")
)

print("\nMissing model-feature summary:")
display(
    latest_snapshot_base["missing_model_feature_count"]
    .value_counts()
    .sort_index()
    .rename_axis("missing_model_feature_count")
    .reset_index(name="stock_count")
)


# Show stocks requiring review.
review_columns = [
    column
    for column in [
        "date",
        "symbol",
        "yf_ticker",
        "company_name",
        "model_ready_v1",
        "missing_model_feature_count",
        "prediction_ready",
        "data_confidence",
    ]
    if column in latest_snapshot_base.columns
]

print("\nStocks requiring limited-confidence review:")
display(
    latest_snapshot_base.loc[
        ~latest_snapshot_base["prediction_ready"],
        review_columns,
    ].sort_values(
        ["data_confidence", "yf_ticker"]
    )
)


if latest_row_count != latest_unique_stocks:
    raise ValueError(
        "The latest cross-section contains duplicate stocks."
    )

if latest_unique_dates != 1:
    raise ValueError(
        "Stocks are not aligned to one common latest date."
    )

print("\nLatest cross-section and readiness validation passed.")

Latest cross-section:
Rows: 100
Unique stocks: 100
Unique latest dates: 1
Latest date: 2026-07-14

Prediction readiness:
Prediction-ready stocks: 90
Limited-confidence stocks: 10

Data-confidence summary:


,data_confidence,stock_count
0,Normal Confidence,90
1,Limited History,8
2,Limited History + Missing Features,2



Missing model-feature summary:


,missing_model_feature_count,stock_count
0,0,98
1,2,2



Stocks requiring limited-confidence review:


,date,symbol,yf_ticker,company_name,model_ready_v1,missing_model_feature_count,prediction_ready,data_confidence
31,2026-07-14,ENRIN,ENRIN.NS,Siemens Energy India Ltd.,0,0,False,Limited History
32,2026-07-14,ETERNAL,ETERNAL.NS,Eternal Ltd.,0,0,False,Limited History
44,2026-07-14,HYUNDAI,HYUNDAI.NS,Hyundai Motor India Ltd.,0,0,False,Limited History
50,2026-07-14,IRFC,IRFC.NS,Indian Railway Finance Corporation Ltd.,0,0,False,Limited History
53,2026-07-14,JIOFIN,JIOFIN.NS,Jio Financial Services Ltd.,0,0,False,Limited History
56,2026-07-14,LODHA,LODHA.NS,Lodha Developers Ltd.,0,0,False,Limited History
61,2026-07-14,MAXHEALTH,MAXHEALTH.NS,Max Healthcare Institute Ltd.,0,0,False,Limited History
62,2026-07-14,MAZDOCK,MAZDOCK.NS,Mazagoan Dock Shipbuilders Ltd.,0,0,False,Limited History
81,2026-07-14,TATACAP,TATACAP.NS,Tata Capital Ltd.,0,2,False,Limited History + Missing Features
88,2026-07-14,TMCV,TMCV.NS,Tata Motors Ltd.,0,2,False,Limited History + Missing Features



Latest cross-section and readiness validation passed.


### Generate model scores, fixed risk bands, and ready-universe ranks.

It creates the parts that must remain consistent with Notebook 07:

Raw outperform score for all 100 stocks

Raw downside score for all 100 stocks

Fixed downside-risk band for reference

Outperform rank among the 90 ready stocks

Downside-risk rank among the 90 ready stocks

It does not create the new quintiles or v2 combined classes yet. First, we verify that the underlying model outputs are correct.

In [5]:
# Generate model scores and ready-universe ranks

rank_based_snapshot = latest_snapshot_base.copy()

X_latest = rank_based_snapshot[MODEL_FEATURES].copy()


def predict_positive_class_probability(
    fitted_model,
    feature_frame: pd.DataFrame,
) -> np.ndarray:
    """
    Return the probability assigned to target class 1.

    This avoids assuming that class 1 is always stored in the second
    predict_proba column without checking the model's fitted classes.
    """

    model_classes = np.asarray(fitted_model.classes_)

    positive_class_locations = np.where(model_classes == 1)[0]

    if len(positive_class_locations) != 1:
        raise ValueError(
            "Expected exactly one fitted class equal to 1. "
            f"Found classes: {model_classes.tolist()}"
        )

    positive_class_index = int(positive_class_locations[0])

    probabilities = fitted_model.predict_proba(feature_frame)

    return probabilities[:, positive_class_index]


# Generate scores for all 100 stocks.
#
# The saved pipelines handle preprocessing and missing-value imputation.
# Limited-confidence stocks receive scores for reference but are not included
# in the normal 90-stock ranks.
rank_based_snapshot["outperform_probability"] = (
    predict_positive_class_probability(
        outperform_model,
        X_latest,
    )
)

rank_based_snapshot["downside_probability"] = (
    predict_positive_class_probability(
        downside_model,
        X_latest,
    )
)


def assign_fixed_downside_risk_band(
    probability: float,
) -> str:
    """
    Retain the original v1 fixed downside-probability band.

    This remains a descriptive absolute-score alert. It will not control
    the v2 combined opportunity-risk classification.
    """

    if pd.isna(probability):
        return "Unavailable"

    if probability < 0.40:
        return "Low Risk"

    if probability < 0.47:
        return "Watch Risk"

    if probability < 0.51:
        return "High Risk"

    return "Very High Risk"


rank_based_snapshot["downside_risk_band"] = (
    rank_based_snapshot["downside_probability"]
    .apply(assign_fixed_downside_risk_band)
)


# Rank only the normal prediction-ready universe.
#
# Outperform rank:
#   Rank 1 = highest opportunity-model score.
#
# Downside-risk rank:
#   Rank 1 = highest downside-risk score.
ready_mask = rank_based_snapshot["prediction_ready"]


rank_based_snapshot[
    "outperform_rank_ready_universe"
] = pd.Series(
    pd.NA,
    index=rank_based_snapshot.index,
    dtype="Int64",
)

rank_based_snapshot[
    "downside_risk_rank_ready_universe"
] = pd.Series(
    pd.NA,
    index=rank_based_snapshot.index,
    dtype="Int64",
)


outperform_ready_ranks = (
    rank_based_snapshot.loc[
        ready_mask,
        "outperform_probability",
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype("Int64")
)

downside_ready_ranks = (
    rank_based_snapshot.loc[
        ready_mask,
        "downside_probability",
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype("Int64")
)


rank_based_snapshot.loc[
    ready_mask,
    "outperform_rank_ready_universe",
] = outperform_ready_ranks

rank_based_snapshot.loc[
    ready_mask,
    "downside_risk_rank_ready_universe",
] = downside_ready_ranks


# Validate score ranges.
probability_columns = [
    "outperform_probability",
    "downside_probability",
]

for column in probability_columns:
    if rank_based_snapshot[column].isna().any():
        raise ValueError(
            f"{column} contains missing predictions."
        )

    if not rank_based_snapshot[column].between(0, 1).all():
        raise ValueError(
            f"{column} contains values outside [0, 1]."
        )


# Validate the two ready-universe ranks.
rank_columns = [
    "outperform_rank_ready_universe",
    "downside_risk_rank_ready_universe",
]

for column in rank_columns:
    ready_ranks = rank_based_snapshot.loc[
        ready_mask,
        column,
    ]

    if ready_ranks.isna().any():
        raise ValueError(
            f"{column} is missing ranks for prediction-ready stocks."
        )

    if ready_ranks.nunique() != ready_stock_count:
        raise ValueError(
            f"{column} does not contain one unique rank per ready stock."
        )

    if ready_ranks.min() != 1:
        raise ValueError(
            f"{column} does not begin at rank 1."
        )

    if ready_ranks.max() != ready_stock_count:
        raise ValueError(
            f"{column} does not end at rank {ready_stock_count}."
        )


# Limited-confidence stocks must remain outside the ranked universe.
if (
    rank_based_snapshot.loc[
        ~ready_mask,
        rank_columns,
    ]
    .notna()
    .any()
    .any()
):
    raise ValueError(
        "Limited-confidence stocks unexpectedly received ready-universe ranks."
    )


print("Model classes:")
print(
    "Outperform model:",
    np.asarray(outperform_model.classes_).tolist(),
)
print(
    "Downside model:",
    np.asarray(downside_model.classes_).tolist(),
)

print("\nPrediction-score ranges:")
for column in probability_columns:
    print(
        f"{column}: "
        f"{rank_based_snapshot[column].min():.6f} "
        f"to {rank_based_snapshot[column].max():.6f}"
    )

print("\nFixed downside-risk band summary:")
display(
    rank_based_snapshot["downside_risk_band"]
    .value_counts()
    .rename_axis("downside_risk_band")
    .reset_index(name="stock_count")
)


ranking_display_columns = [
    column
    for column in [
        "date",
        "symbol",
        "yf_ticker",
        "company_name",
        "outperform_probability",
        "outperform_rank_ready_universe",
        "downside_probability",
        "downside_risk_rank_ready_universe",
        "downside_risk_band",
        "data_confidence",
    ]
    if column in rank_based_snapshot.columns
]


print("\nTop 10 opportunity-ranked stocks:")
display(
    rank_based_snapshot.loc[
        ready_mask,
        ranking_display_columns,
    ]
    .sort_values(
        "outperform_rank_ready_universe"
    )
    .head(10)
)


print("\nTop 10 downside-risk-ranked stocks:")
display(
    rank_based_snapshot.loc[
        ready_mask,
        ranking_display_columns,
    ]
    .sort_values(
        "downside_risk_rank_ready_universe"
    )
    .head(10)
)


print("\nRank validation:")
for column in rank_columns:
    ready_ranks = rank_based_snapshot.loc[
        ready_mask,
        column,
    ]

    print(
        f"{column}: "
        f"{ready_ranks.notna().sum()} ranks, "
        f"range {int(ready_ranks.min())}"
        f"–{int(ready_ranks.max())}, "
        f"{ready_ranks.nunique()} unique"
    )

print("\nModel scoring and rank validation passed.")

Model classes:
Outperform model: [0, 1]
Downside model: [0, 1]

Prediction-score ranges:
outperform_probability: 0.428796 to 0.564380
downside_probability: 0.311189 to 0.709686

Fixed downside-risk band summary:


,downside_risk_band,stock_count
0,Very High Risk,41
1,Watch Risk,29
2,Low Risk,21
3,High Risk,9



Top 10 opportunity-ranked stocks:


,date,symbol,yf_ticker,company_name,outperform_probability,outperform_rank_ready_universe,downside_probability,downside_risk_rank_ready_universe,downside_risk_band,data_confidence
7,2026-07-14,APOLLOHOSP,APOLLOHOSP.NS,Apollo Hospitals Enterprise Ltd.,0.564380,1,0.330996,89,Low Risk,Normal Confidence
35,2026-07-14,GRASIM,GRASIM.NS,Grasim Industries Ltd.,0.537724,2,0.346938,86,Low Risk,Normal Confidence
69,2026-07-14,PIDILITIND,PIDILITIND.NS,Pidilite Industries Ltd.,0.536066,3,0.355195,85,Low Risk,Normal Confidence
98,2026-07-14,WIPRO,WIPRO.NS,Wipro Ltd.,0.527187,4,0.527350,27,Very High Risk,Normal Confidence
48,2026-07-14,INFY,INFY.NS,Infosys Ltd.,0.526448,5,0.632757,4,Very High Risk,Normal Confidence
57,2026-07-14,LT,LT.NS,Larsen & Toubro Ltd.,0.518347,6,0.357374,84,Low Risk,Normal Confidence
87,2026-07-14,TITAN,TITAN.NS,Titan Company Ltd.,0.513995,7,0.386657,75,Low Risk,Normal Confidence
8,2026-07-14,ASIANPAINT,ASIANPAINT.NS,Asian Paints Ltd.,0.512050,8,0.380597,77,Low Risk,Normal Confidence
93,2026-07-14,ULTRACEMCO,ULTRACEMCO.NS,UltraTech Cement Ltd.,0.510594,9,0.364067,82,Low Risk,Normal Confidence
80,2026-07-14,SUNPHARMA,SUNPHARMA.NS,Sun Pharmaceutical Industries Ltd.,0.509656,10,0.376481,78,Low Risk,Normal Confidence



Top 10 downside-risk-ranked stocks:


,date,symbol,yf_ticker,company_name,outperform_probability,outperform_rank_ready_universe,downside_probability,downside_risk_rank_ready_universe,downside_risk_band,data_confidence
97,2026-07-14,VEDL,VEDL.NS,Vedanta Ltd.,0.428796,90,0.709686,1,Very High Risk,Normal Confidence
37,2026-07-14,HCLTECH,HCLTECH.NS,HCL Technologies Ltd.,0.489580,40,0.644713,2,Very High Risk,Normal Confidence
58,2026-07-14,LTM,LTM.NS,LTM Ltd.,0.480629,54,0.644368,3,Very High Risk,Normal Confidence
48,2026-07-14,INFY,INFY.NS,Infosys Ltd.,0.526448,5,0.632757,4,Very High Risk,Normal Confidence
89,2026-07-14,TMPV,TMPV.NS,Tata Motors Passenger Vehicles Ltd.,0.444622,82,0.621657,5,Very High Risk,Normal Confidence
64,2026-07-14,MUTHOOTFIN,MUTHOOTFIN.NS,Muthoot Finance Ltd.,0.449065,81,0.609122,6,Very High Risk,Normal Confidence
85,2026-07-14,TCS,TCS.NS,Tata Consultancy Services Ltd.,0.498468,23,0.604039,7,Very High Risk,Normal Confidence
27,2026-07-14,DLF,DLF.NS,DLF Ltd.,0.468422,77,0.592496,8,Very High Risk,Normal Confidence
91,2026-07-14,TRENT,TRENT.NS,Trent Ltd.,0.472336,72,0.585936,9,Very High Risk,Normal Confidence
43,2026-07-14,HINDZINC,HINDZINC.NS,Hindustan Zinc Ltd.,0.477906,58,0.565261,10,Very High Risk,Normal Confidence



Rank validation:
outperform_rank_ready_universe: 90 ranks, range 1–90, 90 unique
downside_risk_rank_ready_universe: 90 ranks, range 1–90, 90 unique

Model scoring and rank validation passed.


The predictions and rankings match the original snapshot:

        Opportunity scores range from 0.428796 to 0.564380.
        Downside scores range from 0.311189 to 0.709686.
        Both rankings contain exactly 90 unique ranks.
        Limited-confidence stocks remain outside the ranked universe.
        Fixed risk bands are preserved unchanged for v1 reference

### Create opportunity tiers, five risk quintiles, and rank-based combined classes

creates three separate interpretations:

            opportunity_tier from outperform rank.
            downside_risk_quintile for detailed relative risk.
            relative_downside_risk for the simple 50/50 split used by the combined class.

In [6]:
# Create rank-based opportunity and risk classifications

classified_snapshot = rank_based_snapshot.copy()

ready_count = int(
    classified_snapshot["prediction_ready"].sum()
)


# ---------------------------------------------------------
# 1. Opportunity tiers
# ---------------------------------------------------------
#
# Top 20%  = High Opportunity
# Next 30% = Moderate Opportunity
# Bottom 50% = Low Opportunity

high_opportunity_limit = math.ceil(ready_count * 0.20)
moderate_opportunity_limit = math.ceil(ready_count * 0.50)


def assign_opportunity_tier(row: pd.Series) -> str:
    """Assign opportunity tier from outperform rank."""

    if not row["prediction_ready"]:
        return "Limited Confidence"

    rank = int(row["outperform_rank_ready_universe"])

    if rank <= high_opportunity_limit:
        return "High Opportunity"

    if rank <= moderate_opportunity_limit:
        return "Moderate Opportunity"

    return "Low Opportunity"


classified_snapshot["opportunity_tier"] = (
    classified_snapshot.apply(
        assign_opportunity_tier,
        axis=1,
    )
)


# ---------------------------------------------------------
# 2. Detailed downside-risk quintiles
# ---------------------------------------------------------
#
# For 90 ready stocks:
#
# Ranks  1–18 = Q1 — Highest Risk
# Ranks 19–36 = Q2 — High Risk
# Ranks 37–54 = Q3 — Medium Risk
# Ranks 55–72 = Q4 — Lower Risk
# Ranks 73–90 = Q5 — Lowest Risk

risk_quintile_limits = {
    "Q1 — Highest Risk": math.ceil(ready_count * 0.20),
    "Q2 — High Risk": math.ceil(ready_count * 0.40),
    "Q3 — Medium Risk": math.ceil(ready_count * 0.60),
    "Q4 — Lower Risk": math.ceil(ready_count * 0.80),
    "Q5 — Lowest Risk": ready_count,
}


def assign_downside_risk_quintile(row: pd.Series) -> str:
    """Assign detailed relative downside-risk quintile."""

    if not row["prediction_ready"]:
        return "Limited Confidence"

    rank = int(row["downside_risk_rank_ready_universe"])

    if rank <= risk_quintile_limits["Q1 — Highest Risk"]:
        return "Q1 — Highest Risk"

    if rank <= risk_quintile_limits["Q2 — High Risk"]:
        return "Q2 — High Risk"

    if rank <= risk_quintile_limits["Q3 — Medium Risk"]:
        return "Q3 — Medium Risk"

    if rank <= risk_quintile_limits["Q4 — Lower Risk"]:
        return "Q4 — Lower Risk"

    return "Q5 — Lowest Risk"


classified_snapshot["downside_risk_quintile"] = (
    classified_snapshot.apply(
        assign_downside_risk_quintile,
        axis=1,
    )
)


# ---------------------------------------------------------
# 3. Broad relative-risk group
# ---------------------------------------------------------
#
# This broader 50/50 split is used only for the combined class.
#
# Ranks 1–45  = Higher Relative Risk
# Ranks 46–90 = Lower Relative Risk

higher_relative_risk_limit = math.ceil(
    ready_count * 0.50
)


def assign_relative_downside_risk(row: pd.Series) -> str:
    """Assign the broad rank-based risk group."""

    if not row["prediction_ready"]:
        return "Limited Confidence"

    rank = int(row["downside_risk_rank_ready_universe"])

    if rank <= higher_relative_risk_limit:
        return "Higher Relative Risk"

    return "Lower Relative Risk"


classified_snapshot["relative_downside_risk"] = (
    classified_snapshot.apply(
        assign_relative_downside_risk,
        axis=1,
    )
)


# ---------------------------------------------------------
# 4. Rank-based combined opportunity-risk class
# ---------------------------------------------------------

def assign_rank_based_combined_class(
    row: pd.Series,
) -> str:
    """
    Combine opportunity tier with broad relative downside risk.

    Fixed downside-risk bands remain descriptive fields and do not
    control this v2 combined classification.
    """

    if not row["prediction_ready"]:
        return "Limited Confidence - Review"

    opportunity = row["opportunity_tier"]
    relative_risk = row["relative_downside_risk"]

    if (
        opportunity == "High Opportunity"
        and relative_risk == "Lower Relative Risk"
    ):
        return "Attractive Risk-Reward"

    if (
        opportunity == "High Opportunity"
        and relative_risk == "Higher Relative Risk"
    ):
        return "High Opportunity / High Risk"

    if (
        opportunity == "Moderate Opportunity"
        and relative_risk == "Lower Relative Risk"
    ):
        return "Balanced Opportunity"

    if (
        opportunity == "Moderate Opportunity"
        and relative_risk == "Higher Relative Risk"
    ):
        return "Caution"

    if (
        opportunity == "Low Opportunity"
        and relative_risk == "Lower Relative Risk"
    ):
        return "Low Opportunity / Lower Risk"

    return "Unfavourable Risk-Reward"


classified_snapshot["opportunity_risk_class"] = (
    classified_snapshot.apply(
        assign_rank_based_combined_class,
        axis=1,
    )
)


# Record the classification version directly in the snapshot.
classified_snapshot["classification_version"] = SNAPSHOT_VERSION
classified_snapshot["classification_method"] = (
    CLASSIFICATION_METHOD
)


# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

ready_only = classified_snapshot.loc[
    classified_snapshot["prediction_ready"]
].copy()


expected_opportunity_counts = {
    "High Opportunity": high_opportunity_limit,
    "Moderate Opportunity": (
        moderate_opportunity_limit
        - high_opportunity_limit
    ),
    "Low Opportunity": (
        ready_count
        - moderate_opportunity_limit
    ),
}

actual_opportunity_counts = (
    ready_only["opportunity_tier"]
    .value_counts()
    .to_dict()
)

if actual_opportunity_counts != expected_opportunity_counts:
    raise ValueError(
        "Opportunity-tier counts do not match the expected "
        f"rank boundaries.\nExpected: {expected_opportunity_counts}\n"
        f"Actual: {actual_opportunity_counts}"
    )


actual_quintile_counts = (
    ready_only["downside_risk_quintile"]
    .value_counts()
)

if len(actual_quintile_counts) != 5:
    raise ValueError(
        "Expected five downside-risk quintiles."
    )

if not actual_quintile_counts.eq(
    ready_count // 5
).all():
    raise ValueError(
        "The five downside-risk quintiles are not equally sized."
    )


actual_relative_risk_counts = (
    ready_only["relative_downside_risk"]
    .value_counts()
    .to_dict()
)

expected_relative_risk_counts = {
    "Higher Relative Risk": higher_relative_risk_limit,
    "Lower Relative Risk": (
        ready_count - higher_relative_risk_limit
    ),
}

if (
    actual_relative_risk_counts
    != expected_relative_risk_counts
):
    raise ValueError(
        "Relative-risk counts do not match the expected split.\n"
        f"Expected: {expected_relative_risk_counts}\n"
        f"Actual: {actual_relative_risk_counts}"
    )


print("Classification boundaries:")

print("\nOpportunity:")
print(
    f"High Opportunity: ranks 1–{high_opportunity_limit}"
)
print(
    "Moderate Opportunity: ranks "
    f"{high_opportunity_limit + 1}"
    f"–{moderate_opportunity_limit}"
)
print(
    "Low Opportunity: ranks "
    f"{moderate_opportunity_limit + 1}"
    f"–{ready_count}"
)

print("\nDetailed relative downside risk:")
previous_limit = 0

for label, limit in risk_quintile_limits.items():
    print(
        f"{label}: ranks "
        f"{previous_limit + 1}–{limit}"
    )
    previous_limit = limit

print("\nBroad relative downside risk:")
print(
    "Higher Relative Risk: ranks "
    f"1–{higher_relative_risk_limit}"
)
print(
    "Lower Relative Risk: ranks "
    f"{higher_relative_risk_limit + 1}"
    f"–{ready_count}"
)


print("\nOpportunity-tier summary:")
display(
    classified_snapshot["opportunity_tier"]
    .value_counts()
    .rename_axis("opportunity_tier")
    .reset_index(name="stock_count")
)


print("\nDownside-risk quintile summary:")
display(
    classified_snapshot["downside_risk_quintile"]
    .value_counts()
    .rename_axis("downside_risk_quintile")
    .reset_index(name="stock_count")
)


print("\nRelative downside-risk summary:")
display(
    classified_snapshot["relative_downside_risk"]
    .value_counts()
    .rename_axis("relative_downside_risk")
    .reset_index(name="stock_count")
)


print("\nRank-based combined-class summary:")
display(
    classified_snapshot["opportunity_risk_class"]
    .value_counts()
    .rename_axis("opportunity_risk_class")
    .reset_index(name="stock_count")
)


print("\nRank-based classification validation passed.")

Classification boundaries:

Opportunity:
High Opportunity: ranks 1–18
Moderate Opportunity: ranks 19–45
Low Opportunity: ranks 46–90

Detailed relative downside risk:
Q1 — Highest Risk: ranks 1–18
Q2 — High Risk: ranks 19–36
Q3 — Medium Risk: ranks 37–54
Q4 — Lower Risk: ranks 55–72
Q5 — Lowest Risk: ranks 73–90

Broad relative downside risk:
Higher Relative Risk: ranks 1–45
Lower Relative Risk: ranks 46–90

Opportunity-tier summary:


,opportunity_tier,stock_count
0,Low Opportunity,45
1,Moderate Opportunity,27
2,High Opportunity,18
3,Limited Confidence,10



Downside-risk quintile summary:


,downside_risk_quintile,stock_count
0,Q2 — High Risk,18
1,Q1 — Highest Risk,18
2,Q4 — Lower Risk,18
3,Q5 — Lowest Risk,18
4,Q3 — Medium Risk,18
5,Limited Confidence,10



Relative downside-risk summary:


,relative_downside_risk,stock_count
0,Higher Relative Risk,45
1,Lower Relative Risk,45
2,Limited Confidence,10



Rank-based combined-class summary:


,opportunity_risk_class,stock_count
0,Unfavourable Risk-Reward,31
1,Balanced Opportunity,21
2,Low Opportunity / Lower Risk,14
3,Attractive Risk-Reward,10
4,Limited Confidence - Review,10
5,High Opportunity / High Risk,8
6,Caution,6



Rank-based classification validation passed.


This shows The v2 structure is now working:

18 stocks in each detailed risk quintile
45 higher-relative-risk and 45 lower-relative-risk stocks
90 ready stocks distributed across six combined categories
10 limited-confidence stocks kept outside the normal classification

The combined classes are not expected to have equal counts because opportunity rank and downside-risk rank are independent signals.

For example:

31 Unfavourable Risk-Reward

means 31 of the 45 Low Opportunity stocks also fall within the riskier half of the universe. The other 14 Low Opportunity stocks fall in the safer half.

### Recreate the v1 class and compare it with v2.

what changed from v1 to v2

Notebook 07 logic: opportunity tier + fixed probability band
Notebook 09 logic: opportunity tier + relative downside rank

This is important because the purpose of Notebook 09 is not merely to produce another table. We need a documented record showing:

        How many stocks changed category
        Which stocks changed
        Their old and new categories
        Where fixed score bands and relative ranks disagree
        That probabilities and ranks remained unchanged

In [7]:
#  Compare original fixed-band classification with rank-based v2

comparison_snapshot = classified_snapshot.copy()


# ---------------------------------------------------------
# 1. Recreate the original Notebook 07 combined class
# ---------------------------------------------------------

def assign_v1_fixed_band_class(row: pd.Series) -> str:
    """
    Reproduce the Notebook 07 combined classification.

    In v1:
    - Low Risk and Watch Risk were treated as the lower-risk side.
    - High Risk and Very High Risk were treated as the higher-risk side.
    """

    if not row["prediction_ready"]:
        return "Limited Confidence - Review"

    opportunity = row["opportunity_tier"]

    lower_fixed_band_risk = (
        row["downside_risk_band"]
        in {"Low Risk", "Watch Risk"}
    )

    if (
        opportunity == "High Opportunity"
        and lower_fixed_band_risk
    ):
        return "Attractive Risk-Reward"

    if opportunity == "High Opportunity":
        return "High Opportunity / High Risk"

    if (
        opportunity == "Moderate Opportunity"
        and lower_fixed_band_risk
    ):
        return "Balanced Opportunity"

    if opportunity == "Moderate Opportunity":
        return "Caution"

    if (
        opportunity == "Low Opportunity"
        and lower_fixed_band_risk
    ):
        return "Low Opportunity / Lower Risk"

    return "Unfavourable Risk-Reward"


comparison_snapshot[
    "opportunity_risk_class_v1_reference"
] = comparison_snapshot.apply(
    assign_v1_fixed_band_class,
    axis=1,
)


# Rename the existing Notebook 09 result clearly for comparison.
comparison_snapshot[
    "opportunity_risk_class_v2_rank_based"
] = comparison_snapshot["opportunity_risk_class"]


# ---------------------------------------------------------
# 2. Compare broad fixed-band risk with relative rank risk
# ---------------------------------------------------------

def assign_fixed_band_broad_risk(row: pd.Series) -> str:
    """Convert the four fixed bands into the two v1 risk sides."""

    if not row["prediction_ready"]:
        return "Limited Confidence"

    if row["downside_risk_band"] in {
        "Low Risk",
        "Watch Risk",
    }:
        return "Lower Fixed-Band Risk"

    return "Higher Fixed-Band Risk"


comparison_snapshot["fixed_band_broad_risk"] = (
    comparison_snapshot.apply(
        assign_fixed_band_broad_risk,
        axis=1,
    )
)


def normalize_fixed_risk_for_comparison(
    fixed_risk: str,
) -> str:
    """Map fixed-band risk wording to the v2 relative-risk wording."""

    mapping = {
        "Lower Fixed-Band Risk": "Lower Relative Risk",
        "Higher Fixed-Band Risk": "Higher Relative Risk",
        "Limited Confidence": "Limited Confidence",
    }

    return mapping[fixed_risk]


comparison_snapshot[
    "fixed_band_risk_normalized"
] = (
    comparison_snapshot["fixed_band_broad_risk"]
    .map(normalize_fixed_risk_for_comparison)
)


comparison_snapshot["risk_method_agreement"] = np.where(
    comparison_snapshot["fixed_band_risk_normalized"]
    == comparison_snapshot["relative_downside_risk"],
    "Agree",
    "Disagree",
)


# ---------------------------------------------------------
# 3. Identify combined-class changes
# ---------------------------------------------------------

comparison_snapshot["classification_changed"] = (
    comparison_snapshot[
        "opportunity_risk_class_v1_reference"
    ]
    != comparison_snapshot[
        "opportunity_risk_class_v2_rank_based"
    ]
)


ready_comparison = comparison_snapshot.loc[
    comparison_snapshot["prediction_ready"]
].copy()


changed_ready_count = int(
    ready_comparison["classification_changed"].sum()
)

unchanged_ready_count = int(
    (~ready_comparison["classification_changed"]).sum()
)

risk_disagreement_count = int(
    ready_comparison["risk_method_agreement"]
    .eq("Disagree")
    .sum()
)


# A combined class changes exactly when the broad risk side changes,
# because the opportunity tier is identical in v1 and v2.
if changed_ready_count != risk_disagreement_count:
    raise ValueError(
        "Combined-class changes do not match broad risk-method "
        "disagreements."
    )


# ---------------------------------------------------------
# 4. Display comparisons
# ---------------------------------------------------------

print("V1 versus V2 comparison:")
print("Ready stocks:", len(ready_comparison))
print("Unchanged combined classes:", unchanged_ready_count)
print("Changed combined classes:", changed_ready_count)
print("Risk-method disagreements:", risk_disagreement_count)

print(
    "Percentage of ready stocks changed:",
    f"{changed_ready_count / len(ready_comparison):.2%}",
)


print("\nV1 fixed-band combined-class counts:")
display(
    comparison_snapshot[
        "opportunity_risk_class_v1_reference"
    ]
    .value_counts()
    .rename_axis("v1_combined_class")
    .reset_index(name="stock_count")
)


print("\nV2 rank-based combined-class counts:")
display(
    comparison_snapshot[
        "opportunity_risk_class_v2_rank_based"
    ]
    .value_counts()
    .rename_axis("v2_combined_class")
    .reset_index(name="stock_count")
)


print("\nFixed-band versus rank-based risk agreement:")
display(
    comparison_snapshot["risk_method_agreement"]
    .value_counts()
    .rename_axis("risk_method_agreement")
    .reset_index(name="stock_count")
)


print("\nCombined-class transition matrix:")
transition_matrix = pd.crosstab(
    ready_comparison[
        "opportunity_risk_class_v1_reference"
    ],
    ready_comparison[
        "opportunity_risk_class_v2_rank_based"
    ],
    rownames=["v1_fixed_band_class"],
    colnames=["v2_rank_based_class"],
    margins=True,
)

display(transition_matrix)


comparison_display_columns = [
    column
    for column in [
        "date",
        "symbol",
        "yf_ticker",
        "company_name",
        "outperform_probability",
        "outperform_rank_ready_universe",
        "opportunity_tier",
        "downside_probability",
        "downside_risk_rank_ready_universe",
        "downside_risk_band",
        "downside_risk_quintile",
        "fixed_band_broad_risk",
        "relative_downside_risk",
        "opportunity_risk_class_v1_reference",
        "opportunity_risk_class_v2_rank_based",
    ]
    if column in comparison_snapshot.columns
]


print("\nStocks whose combined classification changed:")
display(
    comparison_snapshot.loc[
        comparison_snapshot["classification_changed"],
        comparison_display_columns,
    ]
    .sort_values(
        [
            "opportunity_tier",
            "outperform_rank_ready_universe",
        ],
        na_position="last",
    )
)


print("\nV1-to-v2 comparison validation passed.")

V1 versus V2 comparison:
Ready stocks: 90
Unchanged combined classes: 85
Changed combined classes: 5
Risk-method disagreements: 5
Percentage of ready stocks changed: 5.56%

V1 fixed-band combined-class counts:


,v1_combined_class,stock_count
0,Unfavourable Risk-Reward,27
1,Balanced Opportunity,22
2,Low Opportunity / Lower Risk,18
3,Attractive Risk-Reward,10
4,Limited Confidence - Review,10
5,High Opportunity / High Risk,8
6,Caution,5



V2 rank-based combined-class counts:


,v2_combined_class,stock_count
0,Unfavourable Risk-Reward,31
1,Balanced Opportunity,21
2,Low Opportunity / Lower Risk,14
3,Attractive Risk-Reward,10
4,Limited Confidence - Review,10
5,High Opportunity / High Risk,8
6,Caution,6



Fixed-band versus rank-based risk agreement:


...


Combined-class transition matrix:


v2_rank_based_class,Attractive Risk-Reward,Balanced Opportunity,Caution,High Opportunity / High Risk,Low Opportunity / Lower Risk,Unfavourable Risk-Reward,All
v1_fixed_band_class,,,,,,,
Attractive Risk-Reward,10,0,0,0,0,0,10
Balanced Opportunity,0,21,1,0,0,0,22
Caution,0,0,5,0,0,0,5
High Opportunity / High Risk,0,0,0,8,0,0,8
Low Opportunity / Lower Risk,0,0,0,0,14,4,18
Unfavourable Risk-Reward,0,0,0,0,0,27,27
All,10,21,6,8,14,31,90



Stocks whose combined classification changed:


,date,symbol,yf_ticker,company_name,outperform_probability,outperform_rank_ready_universe,opportunity_tier,downside_probability,downside_risk_rank_ready_universe,downside_risk_band,downside_risk_quintile,fixed_band_broad_risk,relative_downside_risk,opportunity_risk_class_v1_reference,opportunity_risk_class_v2_rank_based
34,2026-07-14,GODREJCP,GODREJCP.NS,Godrej Consumer Products Ltd.,0.480420,55,Low Opportunity,0.462327,42,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Low Opportunity / Lower Risk,Unfavourable Risk-Reward
55,2026-07-14,KOTAKBANK,KOTAKBANK.NS,Kotak Mahindra Bank Ltd.,0.480347,56,Low Opportunity,0.460402,44,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Low Opportunity / Lower Risk,Unfavourable Risk-Reward
13,2026-07-14,BAJFINANCE,BAJFINANCE.NS,Bajaj Finance Ltd.,0.472013,73,Low Opportunity,0.461576,43,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Low Opportunity / Lower Risk,Unfavourable Risk-Reward
11,2026-07-14,BAJAJFINSV,BAJAJFINSV.NS,Bajaj Finserv Ltd.,0.470210,75,Low Opportunity,0.456490,45,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Low Opportunity / Lower Risk,Unfavourable Risk-Reward
96,2026-07-14,VBL,VBL.NS,Varun Beverages Ltd.,0.490055,37,Moderate Opportunity,0.465680,41,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Balanced Opportunity,Caution



V1-to-v2 comparison validation passed.


What the comparison tells us

Only 5 of the 90 ready stocks changed classification:

85 unchanged
5 changed
5.56% changed

So v2 is not radically rewriting the original snapshot. It preserves 94.44% of the classifications and corrects a small set of borderline cases.

Why exactly five stocks changed

The fixed-band v1 system divided the 90 ready stocks like this:

Lower fixed-band risk:  50 stocks
Higher fixed-band risk: 40 stocks

That came from treating Low Risk and Watch Risk as the safer side.

The rank-based v2 system deliberately creates:

Lower Relative Risk:  45 stocks
Higher Relative Risk: 45 stocks

Therefore, the five riskiest stocks from the old 50-stock “lower-risk” group had to move into the higher-relative-risk half.

All five are in the upper half of the current downside ranking even though their raw probabilities remain below the fixed 0.47 High Risk threshold.

This is the exact type of disagreement v2 is designed to handle:

Their absolute score looks like “Watch Risk,” but relative to the other stocks currently available, they belong to the riskier half.

There were no movements in the opposite direction because the fixed-band lower-risk side contained 50 stocks, while the rank-based lower-risk side permits only 45.

### Build and validate final v2 outputs:

We will now create two in-memory outputs:

Main v2 snapshot — intended for future dashboard use.

V1-versus-v2 audit table — preserves exactly how and why classifications changed.

In [8]:
#  Build and validate the final v2 snapshot and audit table

final_snapshot_source = comparison_snapshot.copy()


# The v2 rank-based class becomes the primary class in the final snapshot.
final_snapshot_source["opportunity_risk_class"] = (
    final_snapshot_source[
        "opportunity_risk_class_v2_rank_based"
    ]
)


# ---------------------------------------------------------
# 1. Main v2 prediction snapshot
# ---------------------------------------------------------

preferred_snapshot_columns = [
    "date",
    "symbol",
    "yf_ticker",
    "company_name",
    "close",
    "outperform_probability",
    "outperform_rank_ready_universe",
    "opportunity_tier",
    "downside_probability",
    "downside_risk_rank_ready_universe",
    "downside_risk_band",
    "downside_risk_quintile",
    "relative_downside_risk",
    "opportunity_risk_class",
    "prediction_ready",
    "data_confidence",
    "missing_model_feature_count",
    "classification_version",
    "classification_method",
]

final_snapshot_columns = [
    column
    for column in preferred_snapshot_columns
    if column in final_snapshot_source.columns
]

final_rank_based_snapshot = (
    final_snapshot_source[
        final_snapshot_columns
    ]
    .sort_values(
        [
            "prediction_ready",
            "outperform_rank_ready_universe",
            "yf_ticker",
        ],
        ascending=[False, True, True],
        na_position="last",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 2. V1-versus-v2 audit table
# ---------------------------------------------------------

preferred_audit_columns = [
    "date",
    "symbol",
    "yf_ticker",
    "company_name",
    "outperform_probability",
    "outperform_rank_ready_universe",
    "opportunity_tier",
    "downside_probability",
    "downside_risk_rank_ready_universe",
    "downside_risk_band",
    "downside_risk_quintile",
    "fixed_band_broad_risk",
    "relative_downside_risk",
    "risk_method_agreement",
    "opportunity_risk_class_v1_reference",
    "opportunity_risk_class_v2_rank_based",
    "classification_changed",
    "prediction_ready",
    "data_confidence",
]

audit_columns = [
    column
    for column in preferred_audit_columns
    if column in final_snapshot_source.columns
]

v1_v2_classification_audit = (
    final_snapshot_source[
        audit_columns
    ]
    .sort_values(
        [
            "classification_changed",
            "prediction_ready",
            "downside_risk_rank_ready_universe",
        ],
        ascending=[False, False, True],
        na_position="last",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------
# 3. Final validations
# ---------------------------------------------------------

expected_snapshot_rows = 100
expected_ready_rows = 90
expected_limited_rows = 10


if len(final_rank_based_snapshot) != expected_snapshot_rows:
    raise ValueError(
        "Final snapshot does not contain exactly 100 rows."
    )

if (
    final_rank_based_snapshot["yf_ticker"].nunique()
    != expected_snapshot_rows
):
    raise ValueError(
        "Final snapshot does not contain 100 unique stocks."
    )

if final_rank_based_snapshot["date"].nunique() != 1:
    raise ValueError(
        "Final snapshot contains more than one prediction date."
    )

actual_ready_rows = int(
    final_rank_based_snapshot["prediction_ready"].sum()
)

actual_limited_rows = int(
    (~final_rank_based_snapshot["prediction_ready"]).sum()
)

if actual_ready_rows != expected_ready_rows:
    raise ValueError(
        f"Expected {expected_ready_rows} ready stocks, "
        f"found {actual_ready_rows}."
    )

if actual_limited_rows != expected_limited_rows:
    raise ValueError(
        f"Expected {expected_limited_rows} limited stocks, "
        f"found {actual_limited_rows}."
    )

if (
    final_rank_based_snapshot["opportunity_risk_class"]
    .isna()
    .any()
):
    raise ValueError(
        "Final snapshot contains missing combined classes."
    )

if (
    final_rank_based_snapshot["classification_version"]
    .ne(SNAPSHOT_VERSION)
    .any()
):
    raise ValueError(
        "Unexpected classification version found."
    )

if (
    final_rank_based_snapshot["classification_method"]
    .ne(CLASSIFICATION_METHOD)
    .any()
):
    raise ValueError(
        "Unexpected classification method found."
    )

if len(v1_v2_classification_audit) != expected_snapshot_rows:
    raise ValueError(
        "V1-versus-v2 audit table does not contain 100 rows."
    )


ready_final = final_rank_based_snapshot.loc[
    final_rank_based_snapshot["prediction_ready"]
]

ready_relative_counts = (
    ready_final["relative_downside_risk"]
    .value_counts()
    .to_dict()
)

if ready_relative_counts != {
    "Higher Relative Risk": 45,
    "Lower Relative Risk": 45,
}:
    raise ValueError(
        "Final ready-universe relative-risk split is incorrect."
    )


# ---------------------------------------------------------
# 4. Display final results
# ---------------------------------------------------------

print("Final v2 snapshot:")
print("Shape:", final_rank_based_snapshot.shape)
print(
    "Prediction date:",
    final_rank_based_snapshot["date"].max().date(),
)
print(
    "Unique stocks:",
    final_rank_based_snapshot["yf_ticker"].nunique(),
)
print("Prediction-ready stocks:", actual_ready_rows)
print("Limited-confidence stocks:", actual_limited_rows)


print("\nFinal v2 combined-class summary:")
display(
    final_rank_based_snapshot[
        "opportunity_risk_class"
    ]
    .value_counts()
    .rename_axis("opportunity_risk_class")
    .reset_index(name="stock_count")
)


print("\nDetailed relative-risk summary:")
display(
    final_rank_based_snapshot[
        "downside_risk_quintile"
    ]
    .value_counts()
    .rename_axis("downside_risk_quintile")
    .reset_index(name="stock_count")
)


print("\nAttractive Risk-Reward stocks:")
display(
    final_rank_based_snapshot.loc[
        final_rank_based_snapshot[
            "opportunity_risk_class"
        ].eq("Attractive Risk-Reward")
    ]
    .sort_values("outperform_rank_ready_universe")
)


print("\nHigh Opportunity / High Risk stocks:")
display(
    final_rank_based_snapshot.loc[
        final_rank_based_snapshot[
            "opportunity_risk_class"
        ].eq("High Opportunity / High Risk")
    ]
    .sort_values("outperform_rank_ready_universe")
)


print("\nChanged V1-to-V2 classifications:")
display(
    v1_v2_classification_audit.loc[
        v1_v2_classification_audit[
            "classification_changed"
        ]
    ]
)


print("\nFinal v2 snapshot and audit validation passed.")

Final v2 snapshot:
Shape: (100, 19)
Prediction date: 2026-07-14
Unique stocks: 100
Prediction-ready stocks: 90
Limited-confidence stocks: 10

Final v2 combined-class summary:


,opportunity_risk_class,stock_count
0,Unfavourable Risk-Reward,31
1,Balanced Opportunity,21
2,Low Opportunity / Lower Risk,14
3,Attractive Risk-Reward,10
4,Limited Confidence - Review,10
5,High Opportunity / High Risk,8
6,Caution,6



Detailed relative-risk summary:


,downside_risk_quintile,stock_count
0,Q5 — Lowest Risk,18
1,Q2 — High Risk,18
2,Q1 — Highest Risk,18
3,Q4 — Lower Risk,18
4,Q3 — Medium Risk,18
5,Limited Confidence,10



Attractive Risk-Reward stocks:


...


High Opportunity / High Risk stocks:


,date,symbol,yf_ticker,company_name,close,outperform_probability,outperform_rank_ready_universe,opportunity_tier,downside_probability,downside_risk_rank_ready_universe,downside_risk_band,downside_risk_quintile,relative_downside_risk,opportunity_risk_class,prediction_ready,data_confidence,missing_model_feature_count,classification_version,classification_method
3,2026-07-14,WIPRO,WIPRO.NS,Wipro Ltd.,177.139999,0.527187,4,High Opportunity,0.527350,27,Very High Risk,Q2 — High Risk,Higher Relative Risk,High Opportunity / High Risk,True,Normal Confidence,0,v2,rank_based_relative_downside_risk
4,2026-07-14,INFY,INFY.NS,Infosys Ltd.,1092.900024,0.526448,5,High Opportunity,0.632757,4,Very High Risk,Q1 — Highest Risk,Higher Relative Risk,High Opportunity / High Risk,True,Normal Confidence,0,v2,rank_based_relative_downside_risk
12,2026-07-14,CGPOWER,CGPOWER.NS,CG Power and Industrial Solutions Ltd.,913.849976,0.505924,13,High Opportunity,0.527998,25,Very High Risk,Q2 — High Risk,Higher Relative Risk,High Opportunity / High Risk,True,Normal Confidence,0,v2,rank_based_relative_downside_risk
13,2026-07-14,SIEMENS,SIEMENS.NS,Siemens Ltd.,3510.300049,0.504853,14,High Opportunity,0.485967,37,High Risk,Q3 — Medium Risk,Higher Relative Risk,High Opportunity / High Risk,True,Normal Confidence,0,v2,rank_based_relative_downside_risk
14,2026-07-14,ABB,ABB.NS,ABB India Ltd.,6892.000000,0.504668,15,High Opportunity,0.503660,34,High Risk,Q2 — High Risk,Higher Relative Risk,High Opportunity / High Risk,True,Normal Confidence,0,v2,rank_based_relative_downside_risk
15,2026-07-14,MOTHERSON,MOTHERSON.NS,Samvardhana Motherson International Ltd.,142.059998,0.504273,16,High Opportunity,0.536311,21,Very High Risk,Q2 — High Risk,Higher Relative Risk,High Opportunity / High Risk,True,Normal Confidence,0,v2,rank_based_relative_downside_risk
16,2026-07-14,SOLARINDS,SOLARINDS.NS,Solar Industries India Ltd.,18287.000000,0.503902,17,High Opportunity,0.527297,28,Very High Risk,Q2 — High Risk,Higher Relative Risk,High Opportunity / High Risk,True,Normal Confidence,0,v2,rank_based_relative_downside_risk
17,2026-07-14,CUMMINSIND,CUMMINSIND.NS,Cummins India Ltd.,5493.500000,0.503856,18,High Opportunity,0.494918,36,High Risk,Q2 — High Risk,Higher Relative Risk,High Opportunity / High Risk,True,Normal Confidence,0,v2,rank_based_relative_downside_risk



Changed V1-to-V2 classifications:


,date,symbol,yf_ticker,company_name,outperform_probability,outperform_rank_ready_universe,opportunity_tier,downside_probability,downside_risk_rank_ready_universe,downside_risk_band,downside_risk_quintile,fixed_band_broad_risk,relative_downside_risk,risk_method_agreement,opportunity_risk_class_v1_reference,opportunity_risk_class_v2_rank_based,classification_changed,prediction_ready,data_confidence
0,2026-07-14,VBL,VBL.NS,Varun Beverages Ltd.,0.490055,37,Moderate Opportunity,0.465680,41,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Disagree,Balanced Opportunity,Caution,True,True,Normal Confidence
1,2026-07-14,GODREJCP,GODREJCP.NS,Godrej Consumer Products Ltd.,0.480420,55,Low Opportunity,0.462327,42,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Disagree,Low Opportunity / Lower Risk,Unfavourable Risk-Reward,True,True,Normal Confidence
2,2026-07-14,BAJFINANCE,BAJFINANCE.NS,Bajaj Finance Ltd.,0.472013,73,Low Opportunity,0.461576,43,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Disagree,Low Opportunity / Lower Risk,Unfavourable Risk-Reward,True,True,Normal Confidence
3,2026-07-14,KOTAKBANK,KOTAKBANK.NS,Kotak Mahindra Bank Ltd.,0.480347,56,Low Opportunity,0.460402,44,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Disagree,Low Opportunity / Lower Risk,Unfavourable Risk-Reward,True,True,Normal Confidence
4,2026-07-14,BAJAJFINSV,BAJAJFINSV.NS,Bajaj Finserv Ltd.,0.470210,75,Low Opportunity,0.456490,45,Watch Risk,Q3 — Medium Risk,Lower Fixed-Band Risk,Higher Relative Risk,Disagree,Low Opportunity / Lower Risk,Unfavourable Risk-Reward,True,True,Normal Confidence



Final v2 snapshot and audit validation passed.


This confirms that the final v2 snapshot is internally consistent.

The main result is that the models and rankings did not change. Only the interpretation of relative downside risk changed:

    85 of 90 ready stocks retained their original combined class.
    Five borderline Watch Risk stocks moved into the riskier half based on their cross-sectional ranks.
    The attractive group remains unchanged at 10 stocks.
    The high-opportunity/high-risk group remains unchanged at 8 stocks.
    The revision mainly affects risk interpretation around ranks 41–45.

### Save v2 snapshot, audit trail, metadata, and report

This creates:

        Dated CSV and Parquet snapshots
        Stable latest CSV and Parquet snapshots
        A complete v1-versus-v2 audit table
        Machine-readable JSON metadata
        A human-readable Markdown report

In [9]:
# Save v2 snapshot, audit trail, metadata, and report

snapshot_date = pd.Timestamp(
    final_rank_based_snapshot["date"].max()
)

snapshot_date_string = snapshot_date.strftime("%Y-%m-%d")
generated_at_utc = datetime.now(timezone.utc).isoformat()


# ---------------------------------------------------------
# 1. Define output paths
# ---------------------------------------------------------

dated_snapshot_csv_path = (
    OUTPUT_DIR
    / f"rank_based_prediction_snapshot_{snapshot_date_string}_v2.csv"
)

dated_snapshot_parquet_path = (
    OUTPUT_DIR
    / f"rank_based_prediction_snapshot_{snapshot_date_string}_v2.parquet"
)

latest_snapshot_csv_path = (
    OUTPUT_DIR
    / "latest_rank_based_prediction_snapshot_v2.csv"
)

latest_snapshot_parquet_path = (
    OUTPUT_DIR
    / "latest_rank_based_prediction_snapshot_v2.parquet"
)

audit_csv_path = (
    OUTPUT_DIR
    / f"v1_v2_classification_audit_{snapshot_date_string}_v2.csv"
)

audit_parquet_path = (
    OUTPUT_DIR
    / f"v1_v2_classification_audit_{snapshot_date_string}_v2.parquet"
)

metadata_path = (
    OUTPUT_DIR
    / f"rank_based_prediction_snapshot_metadata_{snapshot_date_string}_v2.json"
)

summary_report_path = (
    OUTPUT_DIR
    / f"rank_based_prediction_snapshot_summary_{snapshot_date_string}_v2.md"
)


# ---------------------------------------------------------
# 2. Save main snapshot and audit outputs
# ---------------------------------------------------------

final_rank_based_snapshot.to_csv(
    dated_snapshot_csv_path,
    index=False,
    float_format="%.8f",
)

final_rank_based_snapshot.to_parquet(
    dated_snapshot_parquet_path,
    index=False,
)

# Stable filenames for dashboard or application consumption.
final_rank_based_snapshot.to_csv(
    latest_snapshot_csv_path,
    index=False,
    float_format="%.8f",
)

final_rank_based_snapshot.to_parquet(
    latest_snapshot_parquet_path,
    index=False,
)


v1_v2_classification_audit.to_csv(
    audit_csv_path,
    index=False,
    float_format="%.8f",
)

v1_v2_classification_audit.to_parquet(
    audit_parquet_path,
    index=False,
)


# ---------------------------------------------------------
# 3. Prepare summary statistics
# ---------------------------------------------------------

combined_class_counts = {
    str(label): int(count)
    for label, count in (
        final_rank_based_snapshot[
            "opportunity_risk_class"
        ]
        .value_counts()
        .items()
    )
}

opportunity_tier_counts = {
    str(label): int(count)
    for label, count in (
        final_rank_based_snapshot[
            "opportunity_tier"
        ]
        .value_counts()
        .items()
    )
}

risk_quintile_counts = {
    str(label): int(count)
    for label, count in (
        final_rank_based_snapshot[
            "downside_risk_quintile"
        ]
        .value_counts()
        .items()
    )
}

relative_risk_counts = {
    str(label): int(count)
    for label, count in (
        final_rank_based_snapshot[
            "relative_downside_risk"
        ]
        .value_counts()
        .items()
    )
}

fixed_band_counts = {
    str(label): int(count)
    for label, count in (
        final_rank_based_snapshot[
            "downside_risk_band"
        ]
        .value_counts()
        .items()
    )
}


changed_rows = (
    v1_v2_classification_audit.loc[
        v1_v2_classification_audit[
            "classification_changed"
        ]
    ]
    .copy()
    .sort_values(
        "downside_risk_rank_ready_universe"
    )
)


changed_stock_records = []

for _, row in changed_rows.iterrows():
    changed_stock_records.append(
        {
            "symbol": str(row["symbol"]),
            "yf_ticker": str(row["yf_ticker"]),
            "company_name": str(row["company_name"]),
            "outperform_probability": round(
                float(row["outperform_probability"]),
                8,
            ),
            "outperform_rank": int(
                row["outperform_rank_ready_universe"]
            ),
            "opportunity_tier": str(
                row["opportunity_tier"]
            ),
            "downside_probability": round(
                float(row["downside_probability"]),
                8,
            ),
            "downside_risk_rank": int(
                row["downside_risk_rank_ready_universe"]
            ),
            "fixed_downside_risk_band": str(
                row["downside_risk_band"]
            ),
            "downside_risk_quintile": str(
                row["downside_risk_quintile"]
            ),
            "v1_class": str(
                row[
                    "opportunity_risk_class_v1_reference"
                ]
            ),
            "v2_class": str(
                row[
                    "opportunity_risk_class_v2_rank_based"
                ]
            ),
        }
    )


changed_ready_count = int(
    changed_rows["prediction_ready"].sum()
)

ready_count = int(
    final_rank_based_snapshot[
        "prediction_ready"
    ].sum()
)

changed_percentage = (
    changed_ready_count / ready_count * 100
)


# ---------------------------------------------------------
# 4. Create machine-readable metadata
# ---------------------------------------------------------

metadata = {
    "notebook": "09_rank_based_prediction_snapshot.ipynb",
    "notebook_version": NOTEBOOK_VERSION,
    "snapshot_version": SNAPSHOT_VERSION,
    "snapshot_date": snapshot_date_string,
    "generated_at_utc": generated_at_utc,
    "classification_method": CLASSIFICATION_METHOD,

    "version_lineage": {
        "v1_notebook": "07_latest_prediction_snapshot.ipynb",
        "validation_notebook": "08_historical_signal_backtest.ipynb",
        "v2_notebook": "09_rank_based_prediction_snapshot.ipynb",
        "v1_primary_risk_input": (
            "Fixed downside-probability bands"
        ),
        "v2_primary_risk_input": (
            "Cross-sectional downside-risk rank"
        ),
    },

    "input_files": {
        "feature_dataset": (
            FEATURE_DATA_PATH
            .relative_to(PROJECT_ROOT)
            .as_posix()
        ),
        "outperform_model": (
            OUTPERFORM_MODEL_PATH
            .relative_to(PROJECT_ROOT)
            .as_posix()
        ),
        "downside_model": (
            DOWNSIDE_MODEL_PATH
            .relative_to(PROJECT_ROOT)
            .as_posix()
        ),
    },

    "model_configuration": {
        "feature_count": int(len(MODEL_FEATURES)),
        "models_use_identical_feature_order": bool(
            same_feature_order
        ),
        "outperform_target": (
            "target_outperform_nifty50_20d"
        ),
        "downside_target": (
            "target_big_downside_10pct_20d"
        ),
        "probability_interpretation": (
            "Raw model scores are primarily used for ranking. "
            "They should not be interpreted as fully calibrated "
            "real-world probabilities."
        ),
    },

    "universe": {
        "total_stocks": int(
            len(final_rank_based_snapshot)
        ),
        "prediction_ready_stocks": ready_count,
        "limited_confidence_stocks": int(
            (
                ~final_rank_based_snapshot[
                    "prediction_ready"
                ]
            ).sum()
        ),
        "normal_confidence_stocks": int(
            final_rank_based_snapshot[
                "data_confidence"
            ]
            .eq("Normal Confidence")
            .sum()
        ),
        "limited_history_stocks": int(
            final_rank_based_snapshot[
                "data_confidence"
            ]
            .eq("Limited History")
            .sum()
        ),
        "limited_history_missing_feature_stocks": int(
            final_rank_based_snapshot[
                "data_confidence"
            ]
            .eq(
                "Limited History + Missing Features"
            )
            .sum()
        ),
    },

    "classification_rules": {
        "opportunity_tiers": {
            "High Opportunity": "Outperform ranks 1–18",
            "Moderate Opportunity": (
                "Outperform ranks 19–45"
            ),
            "Low Opportunity": "Outperform ranks 46–90",
        },
        "fixed_downside_risk_bands": {
            "Low Risk": "Probability below 0.40",
            "Watch Risk": (
                "Probability from 0.40 to below 0.47"
            ),
            "High Risk": (
                "Probability from 0.47 to below 0.51"
            ),
            "Very High Risk": (
                "Probability 0.51 or higher"
            ),
            "usage": (
                "Descriptive absolute-score alert only. "
                "Does not control the v2 combined class."
            ),
        },
        "downside_risk_quintiles": {
            "Q1 — Highest Risk": "Ranks 1–18",
            "Q2 — High Risk": "Ranks 19–36",
            "Q3 — Medium Risk": "Ranks 37–54",
            "Q4 — Lower Risk": "Ranks 55–72",
            "Q5 — Lowest Risk": "Ranks 73–90",
        },
        "broad_relative_risk": {
            "Higher Relative Risk": "Ranks 1–45",
            "Lower Relative Risk": "Ranks 46–90",
            "usage": (
                "Controls the v2 combined "
                "opportunity-risk classification."
            ),
        },
    },

    "snapshot_counts": {
        "opportunity_tiers": opportunity_tier_counts,
        "fixed_downside_risk_bands": fixed_band_counts,
        "downside_risk_quintiles": risk_quintile_counts,
        "relative_downside_risk": relative_risk_counts,
        "combined_classes": combined_class_counts,
    },

    "v1_v2_comparison": {
        "ready_stocks_compared": ready_count,
        "unchanged_classifications": int(
            ready_count - changed_ready_count
        ),
        "changed_classifications": changed_ready_count,
        "changed_percentage": round(
            changed_percentage,
            4,
        ),
        "interpretation": (
            "V2 preserved the original classification for "
            "94.44% of prediction-ready stocks. Five stocks "
            "classified as lower fixed-band risk were in the "
            "riskier half of the current cross-section and were "
            "therefore moved to higher-risk combined classes."
        ),
        "changed_stocks": changed_stock_records,
    },

    "historical_evidence_from_notebook_08": {
        "evaluation_period": "2025 onward",
        "common_months": 18,
        "comparison": (
            "Rank-based Attractive Risk-Reward minus "
            "Unfavourable Risk-Reward"
        ),
        "future_return_difference_percentage_points": -0.02,
        "future_return_95_ci": [-2.25, 1.99],
        "worst_path_difference_percentage_points": 1.84,
        "worst_path_95_ci": [0.73, 3.14],
        "downside_5pct_rate_difference_percentage_points": -13.70,
        "downside_5pct_rate_95_ci": [-24.59, -3.98],
        "downside_10pct_rate_difference_percentage_points": -9.60,
        "downside_10pct_rate_95_ci": [-19.70, -1.99],
        "conclusion": (
            "The rank-based combined classification showed "
            "statistically meaningful downside-risk separation "
            "but no statistically reliable final-return advantage."
        ),
    },

    "limitations": [
        (
            "The system is a risk-intelligence and screening "
            "framework, not a buy or sell recommendation."
        ),
        (
            "The outperform model has not demonstrated a "
            "statistically conclusive return-alpha advantage."
        ),
        (
            "Raw predict_proba outputs have not been fully "
            "calibrated as real-world probabilities."
        ),
        (
            "The cleaner historical test contains only "
            "18 monthly periods."
        ),
        (
            "The historical signal test does not include "
            "transaction costs, slippage, turnover, liquidity, "
            "position sizing, or portfolio constraints."
        ),
        (
            "Limited-confidence stocks receive model scores for "
            "reference but are excluded from normal-universe ranks."
        ),
    ],

    "output_files": {
        "dated_snapshot_csv": (
            dated_snapshot_csv_path.name
        ),
        "dated_snapshot_parquet": (
            dated_snapshot_parquet_path.name
        ),
        "latest_snapshot_csv": (
            latest_snapshot_csv_path.name
        ),
        "latest_snapshot_parquet": (
            latest_snapshot_parquet_path.name
        ),
        "classification_audit_csv": audit_csv_path.name,
        "classification_audit_parquet": (
            audit_parquet_path.name
        ),
        "metadata_json": metadata_path.name,
        "summary_markdown": summary_report_path.name,
    },
}


with metadata_path.open(
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        metadata,
        metadata_file,
        indent=2,
        ensure_ascii=False,
    )


# ---------------------------------------------------------
# 5. Create human-readable Markdown summary
# ---------------------------------------------------------

summary_lines = [
    "# Rank-Based Prediction Snapshot — Version 2",
    "",
    f"**Snapshot date:** {snapshot_date_string}",
    "",
    f"**Generated at:** {generated_at_utc}",
    "",
    "## Purpose",
    "",
    (
        "This report documents version 2 of the MarketGuard "
        "latest prediction snapshot."
    ),
    "",
    (
        "Version 1 used fixed downside-probability bands to "
        "control the combined opportunity-risk classification. "
        "Version 2 preserves those bands as descriptive alerts "
        "but uses cross-sectional downside-risk rank for the "
        "primary combined class."
    ),
    "",
    "## Version Lineage",
    "",
    "| Stage | Notebook | Main role |",
    "|---|---|---|",
    (
        "| v1 | `07_latest_prediction_snapshot.ipynb` "
        "| Fixed-band latest snapshot |"
    ),
    (
        "| Validation | `08_historical_signal_backtest.ipynb` "
        "| Historical fixed-versus-rank evaluation |"
    ),
    (
        "| v2 | `09_rank_based_prediction_snapshot.ipynb` "
        "| Rank-based latest snapshot |"
    ),
    "",
    "## Latest Universe",
    "",
    f"- Total stocks: **{len(final_rank_based_snapshot)}**",
    f"- Prediction-ready stocks: **{ready_count}**",
    (
        "- Limited-confidence stocks: "
        f"**{len(final_rank_based_snapshot) - ready_count}**"
    ),
    f"- Model features: **{len(MODEL_FEATURES)}**",
    "",
    "## V2 Combined-Class Counts",
    "",
    "| Combined class | Stock count |",
    "|---|---:|",
]

for label, count in combined_class_counts.items():
    summary_lines.append(
        f"| {label} | {count} |"
    )


summary_lines.extend(
    [
        "",
        "## V1 Versus V2",
        "",
        (
            f"- Unchanged ready-stock classifications: "
            f"**{ready_count - changed_ready_count}**"
        ),
        (
            f"- Changed ready-stock classifications: "
            f"**{changed_ready_count}**"
        ),
        (
            f"- Percentage changed: "
            f"**{changed_percentage:.2f}%**"
        ),
        "",
        (
            "All five changes occurred because the stocks were "
            "inside the riskier half of the current universe even "
            "though their fixed downside score remained in the "
            "`Watch Risk` band."
        ),
        "",
        "### Changed Stocks",
        "",
        (
            "| Symbol | Downside score | Risk rank | "
            "Risk quintile | V1 class | V2 class |"
        ),
        "|---|---:|---:|---|---|---|",
    ]
)


for record in changed_stock_records:
    summary_lines.append(
        "| "
        f"{record['symbol']} | "
        f"{record['downside_probability']:.6f} | "
        f"{record['downside_risk_rank']} | "
        f"{record['downside_risk_quintile']} | "
        f"{record['v1_class']} | "
        f"{record['v2_class']} |"
    )


summary_lines.extend(
    [
        "",
        "## Historical Evidence",
        "",
        (
            "Notebook 08 evaluated the rank-based classification "
            "over 18 monthly observations from 2025 onward."
        ),
        "",
        (
            "Compared with the Unfavourable Risk-Reward group, "
            "the Attractive Risk-Reward group had:"
        ),
        "",
        (
            "- **0.02 percentage points lower final return**, "
            "with a confidence interval crossing zero"
        ),
        (
            "- **1.84 percentage points shallower worst-path "
            "returns**"
        ),
        (
            "- **13.70 percentage points fewer 5% downside events**"
        ),
        (
            "- **9.60 percentage points fewer 10% downside events**"
        ),
        "",
        (
            "The evidence supports the classification primarily "
            "as a downside-risk screen. It does not establish a "
            "reliable final-return or alpha advantage."
        ),
        "",
        "## Interpretation",
        "",
        (
            "- `downside_risk_band` is an absolute score-based "
            "descriptive alert."
        ),
        (
            "- `downside_risk_quintile` gives detailed relative "
            "risk from Q1 to Q5."
        ),
        (
            "- `relative_downside_risk` provides the broad "
            "higher/lower split used by the combined class."
        ),
        (
            "- `opportunity_risk_class` is a screening label, "
            "not an investment recommendation."
        ),
        "",
        "## Limitations",
        "",
        (
            "- Raw model probabilities are not fully calibrated "
            "real-world probabilities."
        ),
        (
            "- The clean historical test contains only 18 months."
        ),
        (
            "- Transaction costs, slippage, turnover, liquidity, "
            "position sizing, and portfolio constraints are not "
            "included."
        ),
        (
            "- Limited-confidence stocks remain outside the "
            "normal ranked universe."
        ),
        "",
    ]
)


summary_report_path.write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)


# ---------------------------------------------------------
# 6. Confirm that all artifacts were created
# ---------------------------------------------------------

saved_artifact_paths = [
    dated_snapshot_csv_path,
    dated_snapshot_parquet_path,
    latest_snapshot_csv_path,
    latest_snapshot_parquet_path,
    audit_csv_path,
    audit_parquet_path,
    metadata_path,
    summary_report_path,
]


missing_saved_artifacts = [
    path
    for path in saved_artifact_paths
    if not path.exists()
]

if missing_saved_artifacts:
    raise FileNotFoundError(
        "The following output artifacts were not created:\n"
        + "\n".join(
            str(path)
            for path in missing_saved_artifacts
        )
    )


print("Saved v2 artifacts:")

for path in saved_artifact_paths:
    print(
        f"{path.name:<70} "
        f"{path.stat().st_size:,} bytes"
    )

print("\nV2 snapshot, audit trail, metadata, and report saved.")

Saved v2 artifacts:
rank_based_prediction_snapshot_2026-07-14_v2.csv                       25,401 bytes
rank_based_prediction_snapshot_2026-07-14_v2.parquet                   20,618 bytes
latest_rank_based_prediction_snapshot_v2.csv                           25,401 bytes
latest_rank_based_prediction_snapshot_v2.parquet                       20,618 bytes
v1_v2_classification_audit_2026-07-14_v2.csv                           26,031 bytes
v1_v2_classification_audit_2026-07-14_v2.parquet                       20,246 bytes
rank_based_prediction_snapshot_metadata_2026-07-14_v2.json             9,053 bytes
rank_based_prediction_snapshot_summary_2026-07-14_v2.md                3,542 bytes

V2 snapshot, audit trail, metadata, and report saved.


### Reload and validate saved artifacts

In [11]:
#  Independently reload and validate saved v2 artifacts

reloaded_dated_csv = pd.read_csv(
    dated_snapshot_csv_path,
    parse_dates=["date"],
)

reloaded_dated_parquet = pd.read_parquet(
    dated_snapshot_parquet_path,
)

reloaded_latest_csv = pd.read_csv(
    latest_snapshot_csv_path,
    parse_dates=["date"],
)

reloaded_latest_parquet = pd.read_parquet(
    latest_snapshot_parquet_path,
)

reloaded_audit_csv = pd.read_csv(
    audit_csv_path,
    parse_dates=["date"],
)

reloaded_audit_parquet = pd.read_parquet(
    audit_parquet_path,
)

with metadata_path.open(
    "r",
    encoding="utf-8",
) as metadata_file:
    reloaded_metadata = json.load(metadata_file)

reloaded_summary_markdown = summary_report_path.read_text(
    encoding="utf-8",
)


# ---------------------------------------------------------
# Normalize dates before comparing formats
# ---------------------------------------------------------

for frame in [
    reloaded_dated_csv,
    reloaded_dated_parquet,
    reloaded_latest_csv,
    reloaded_latest_parquet,
    reloaded_audit_csv,
    reloaded_audit_parquet,
]:
    frame["date"] = pd.to_datetime(frame["date"])


# ---------------------------------------------------------
# Validate main snapshot files
# ---------------------------------------------------------

snapshot_frames = {
    "dated CSV": reloaded_dated_csv,
    "dated Parquet": reloaded_dated_parquet,
    "latest CSV": reloaded_latest_csv,
    "latest Parquet": reloaded_latest_parquet,
}

for label, frame in snapshot_frames.items():
    if len(frame) != 100:
        raise ValueError(
            f"{label} contains {len(frame)} rows instead of 100."
        )

    if frame["yf_ticker"].nunique() != 100:
        raise ValueError(
            f"{label} does not contain 100 unique stocks."
        )

    if frame["date"].nunique() != 1:
        raise ValueError(
            f"{label} contains more than one prediction date."
        )

    if int(frame["prediction_ready"].sum()) != 90:
        raise ValueError(
            f"{label} does not contain 90 prediction-ready stocks."
        )

    if (
        frame["classification_version"]
        .ne("v2")
        .any()
    ):
        raise ValueError(
            f"{label} contains an unexpected version."
        )

    if (
        frame["classification_method"]
        .ne("rank_based_relative_downside_risk")
        .any()
    ):
        raise ValueError(
            f"{label} contains an unexpected classification method."
        )


# ---------------------------------------------------------
# Compare dated and stable latest files
# ---------------------------------------------------------

comparison_columns = [
    "yf_ticker",
    "outperform_probability",
    "outperform_rank_ready_universe",
    "downside_probability",
    "downside_risk_rank_ready_universe",
    "downside_risk_quintile",
    "relative_downside_risk",
    "opportunity_risk_class",
    "prediction_ready",
    "data_confidence",
]

def prepare_for_comparison(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    """
    Normalize CSV and Parquet representations before comparison.

    CSV stores missing nullable-integer ranks as float NaN, while
    Parquet preserves pandas <NA>. Converting rank columns to ordinary
    float64 gives both formats the same missing-value representation.
    """

    prepared = (
        frame[comparison_columns]
        .copy()
        .sort_values("yf_ticker")
        .reset_index(drop=True)
    )

    numeric_columns = [
        "outperform_probability",
        "outperform_rank_ready_universe",
        "downside_probability",
        "downside_risk_rank_ready_universe",
    ]

    for column in numeric_columns:
        prepared[column] = (
            pd.to_numeric(
                prepared[column],
                errors="coerce",
            )
            .astype("float64")
        )

    text_columns = [
        "yf_ticker",
        "downside_risk_quintile",
        "relative_downside_risk",
        "opportunity_risk_class",
        "data_confidence",
    ]

    for column in text_columns:
        prepared[column] = prepared[column].astype("string")

    prepared["prediction_ready"] = (
        prepared["prediction_ready"].astype("bool")
    )

    return prepared

# ---------------------------------------------------------
# Validate audit files
# ---------------------------------------------------------

for label, frame in {
    "audit CSV": reloaded_audit_csv,
    "audit Parquet": reloaded_audit_parquet,
}.items():
    if len(frame) != 100:
        raise ValueError(
            f"{label} contains {len(frame)} rows instead of 100."
        )

    changed_count = int(
        frame["classification_changed"].sum()
    )

    if changed_count != 5:
        raise ValueError(
            f"{label} contains {changed_count} changed "
            "classifications instead of 5."
        )


# ---------------------------------------------------------
# Validate metadata and Markdown documentation
# ---------------------------------------------------------

if reloaded_metadata["snapshot_version"] != "v2":
    raise ValueError(
        "Metadata contains an unexpected snapshot version."
    )

if (
    reloaded_metadata["snapshot_date"]
    != snapshot_date_string
):
    raise ValueError(
        "Metadata snapshot date does not match the saved snapshot."
    )

if (
    reloaded_metadata["universe"]["total_stocks"]
    != 100
):
    raise ValueError(
        "Metadata total-stock count is incorrect."
    )

if (
    reloaded_metadata[
        "v1_v2_comparison"
    ]["changed_classifications"]
    != 5
):
    raise ValueError(
        "Metadata changed-classification count is incorrect."
    )

required_report_sections = [
    "# Rank-Based Prediction Snapshot — Version 2",
    "## Version Lineage",
    "## V1 Versus V2",
    "## Historical Evidence",
    "## Limitations",
]

missing_report_sections = [
    section
    for section in required_report_sections
    if section not in reloaded_summary_markdown
]

if missing_report_sections:
    raise ValueError(
        "Summary report is missing required sections:\n"
        + "\n".join(missing_report_sections)
    )


print("Reloaded snapshot files:")
for label, frame in snapshot_frames.items():
    print(
        f"{label:<16} "
        f"shape={frame.shape}, "
        f"stocks={frame['yf_ticker'].nunique()}, "
        f"ready={int(frame['prediction_ready'].sum())}"
    )

print("\nReloaded audit files:")
print(
    "Audit CSV:",
    reloaded_audit_csv.shape,
    "changed:",
    int(
        reloaded_audit_csv[
            "classification_changed"
        ].sum()
    ),
)
print(
    "Audit Parquet:",
    reloaded_audit_parquet.shape,
    "changed:",
    int(
        reloaded_audit_parquet[
            "classification_changed"
        ].sum()
    ),
)

print("\nMetadata:")
print("Version:", reloaded_metadata["snapshot_version"])
print("Snapshot date:", reloaded_metadata["snapshot_date"])
print(
    "Classification method:",
    reloaded_metadata["classification_method"],
)

print("\nSummary report characters:")
print(len(reloaded_summary_markdown))

print("\nIndependent saved-artifact validation passed.")

Reloaded snapshot files:
dated CSV        shape=(100, 19), stocks=100, ready=90
dated Parquet    shape=(100, 19), stocks=100, ready=90
latest CSV       shape=(100, 19), stocks=100, ready=90
latest Parquet   shape=(100, 19), stocks=100, ready=90

Reloaded audit files:
Audit CSV: (100, 19) changed: 5
Audit Parquet: (100, 19) changed: 5

Metadata:
Version: v2
Snapshot date: 2026-07-14
Classification method: rank_based_relative_downside_risk

Summary report characters:
3447

Independent saved-artifact validation passed.


Notebook 09 is now technically complete.

We have verified that:

    All four snapshot files reload correctly.
    CSV and Parquet contain the same predictions and classifications.
    All files contain 100 unique stocks and 90 prediction-ready stocks.
    The audit files consistently identify the same five v1-to-v2 changes.
    Metadata and Markdown documentation are present and internally consistent.
    The stable latest files match the dated files.

The remaining question is historical:

Does Notebook 09’s exact production logic reproduce the risk separation observed in Notebook 08?